# 🥗 SmartFood Chatbot Server (Colab + ngrok) - Bản Tối Ưu cho App SmartFood
Notebook này phục vụ chạy API Chatbot dinh dưỡng bằng mô hình LoRA Qwen2.5-3B + RAG Knowledge Base.
- Chạy trên GPU T4 miễn phí của Google Colab.
- Mở API ra ngoài Internet qua ngrok để Backend Node.js của SmartFood kết nối trực tiếp.
- **Tối ưu hóa**: Tự động tìm đường dẫn file trên Google Drive, hỗ trợ CORS, hỗ trợ ngữ cảnh đa lượt chat (`history`), tự ngắt tunnel cũ tránh lỗi trùng lặp.

**Chuẩn bị (làm một lần):**
1. Menu Colab: **Runtime** -> **Change runtime type** -> Chọn **T4 GPU** -> Bấm **Save**.
2. Cột trái Colab -> Biểu tượng chìa khóa (**Secrets**) -> Thêm 2 secret và bật *Notebook access*:
   - `NGROK_TOKEN`: Token lấy tại https://dashboard.ngrok.com
   - `CHATBOT_API_KEY`: Chuỗi bí mật bạn tự đặt (ví dụ: `smartfood_secret_key_2026`)
   - `NGROK_DOMAIN` (tùy chọn): Tên miền tĩnh ngrok miễn phí nếu bạn có đăng ký.

## Bước 1: Kết nối Google Drive & Tự động tìm gói chatbot
*(Notebook sẽ tự động quét trong Drive để tìm file `chatbot_v3.zip` và thư mục `lora_nutri_v3`)*

In [ ]:
from google.colab import drive
import os
import glob
from pathlib import Path

# 1. Mount Google Drive
drive.mount("/content/drive")

# 2. Tìm kiếm tự động file chatbot_v3.zip và lora_nutri_v3 trên Drive
def find_path(patterns):
    for p in patterns:
        matches = glob.glob(p, recursive=True)
        if matches and os.path.exists(matches[0]):
            return matches[0]
    return None

candidate_zips = [
    "/content/drive/MyDrive/food_project/chatbot_v2/chatbot_v3.zip",
    "/content/drive/MyDrive/chatbot_v2/chatbot_v3.zip",
    "/content/drive/MyDrive/chatbot_v3.zip",
    "/content/drive/MyDrive/**/chatbot_v3.zip",
    "/content/chatbot_v3.zip",
]

candidate_loras = [
    "/content/drive/MyDrive/food_project/chatbot_v2/chatbot_v3_out/lora_nutri_v3",
    "/content/drive/MyDrive/chatbot_v2/chatbot_v3_out/lora_nutri_v3",
    "/content/drive/MyDrive/**/lora_nutri_v3",
    "/content/drive/MyDrive/lora_nutri_v3",
    "/content/lora_nutri_v3",
]

print("Đang quét Drive để định vị dữ liệu...")
ZIP = find_path(candidate_zips)
LORA = find_path(candidate_loras)

if not ZIP:
    raise FileNotFoundError("Không tìm thấy file chatbot_v3.zip trên Drive. Vui lòng tải file này lên Google Drive rồi chạy lại!")
if not LORA:
    raise FileNotFoundError("Không tìm thấy thư mục lora_nutri_v3 trên Drive. Vui lòng kiểm tra lại thư mục mô hình!")

print(f"✔ Đã tìm thấy ZIP : {ZIP}")
print(f"✔ Đã tìm thấy LORA: {LORA}")

# 3. Giải nén vào /content/chatbot
!unzip -q -o "{ZIP}" -d /content
%cd /content/chatbot
!ls -lh

## Bước 2: Cài đặt các thư viện cần thiết (khoảng 2-3 phút)

In [ ]:
%%capture
!pip install unsloth
!pip install chromadb sentence-transformers fastapi uvicorn pyngrok pydantic requests

## Bước 3: Dựng chỉ mục cơ sở tri thức RAG (Knowledge Base)

In [ ]:
%cd /content/chatbot
!python build_kb.py
!sed -i 's/"kb"/"kb_food"/g' index_kb.py rag_chat.py
!python index_kb.py 2>&1 | tail -n 5

## Bước 4: Nạp mô hình LoRA, Embedding và hàm suy luận

In [ ]:
import sys, os, threading
sys.path.insert(0, "/content/chatbot")

import torch
from unsloth import FastLanguageModel
import chromadb
from sentence_transformers import SentenceTransformer
from common import EMBED_MODEL, build_ctx, build_messages, load_nutrition, clean_num

MAX_LEN = 4096
print(f"Đang nạp mô hình LoRA từ: {LORA} ...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=LORA,
    max_seq_length=MAX_LEN,
    load_in_4bit=True
)
FastLanguageModel.for_inference(model)

print("Đang nạp mô hình Embedding và ChromaDB...")
embedder = SentenceTransformer(EMBED_MODEL, device="cuda")
client = chromadb.PersistentClient("/content/chatbot/chroma_db")
col = client.get_collection("kb_food")
FOODS = load_nutrition("nutrition.csv").to_dict("records")
GPU_LOCK = threading.Lock()   # Khóa GPU để xử lý tuần tự từng request

def retrieve(question, k=4):
    q = embedder.encode([question], normalize_embeddings=True).tolist()
    res = col.query(query_embeddings=q, n_results=k)
    return res["documents"][0] if res["documents"] else []

def generate(msgs, max_new_tokens=250):
    text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    # Giới hạn an toàn để không bao giờ bị tràn RoPE tensor (fix lỗi 2048 vs 2068)
    max_input_len = MAX_LEN - max_new_tokens - 32
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=max_input_len
    ).to("cuda")
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.3,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

print(f"✔ Mô hình và RAG đã sẵn sàng! Số đoạn tri thức: {col.count()}")

## Bước 5: Định nghĩa API FastAPI phục vụ SmartFood App
*(Đã cấu hình CORS, hỗ trợ hồ sơ người dùng, các bữa ăn hôm nay và ngữ cảnh lịch sử hội thoại)*

In [ ]:
import random, datetime, secrets
from typing import List, Optional, Literal, Dict, Any
from fastapi import FastAPI, Header, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel, Field

class Profile(BaseModel):
    tuoi: int = Field(ge=5, le=120)
    gioi_tinh: Literal["nam", "nu"]
    can_nang_kg: float = Field(ge=10, le=350)
    chieu_cao_cm: float = Field(ge=50, le=250)
    muc_van_dong: Literal["it", "nhe", "vua", "nhieu", "rat_nhieu"]
    muc_tieu: Literal["giam_can", "duy_tri", "tang_co", "can_bang"]
    mang_thai: bool = False
    benh_ly: bool = False

class Meal(BaseModel):
    bua: Literal["sang", "trua", "toi", "phu"]
    mon: List[str] = Field(default_factory=list, max_length=20)
    kcal: float = Field(default=0, ge=0, le=10000)
    protein_g: float = Field(default=0, ge=0, le=1000)
    chat_beo_g: float = Field(default=0, ge=0, le=1000)
    carb_g: float = Field(default=0, ge=0, le=2000)
    chat_xo_g: float = Field(default=0, ge=0, le=500)

class LastItem(BaseModel):
    ten: str = Field(max_length=100)
    gram: float = Field(default=0, ge=0, le=5000)
    kcal: float = Field(default=0, ge=0, le=10000)
    protein_g: float = Field(default=0, ge=0, le=1000)

class ChatIn(BaseModel):
    question: str = Field(min_length=1, max_length=1000)
    profile: Optional[Profile] = None
    meals: List[Meal] = Field(default_factory=list, max_length=20)
    last_meal: Optional[List[LastItem]] = Field(default=None, max_length=20)
    history: Optional[List[Dict[str, str]]] = None

def norm_ctx(b: ChatIn):
    profile = None
    if b.profile:
        p = b.profile.model_dump()
        p["can_nang_kg"] = clean_num(round(p["can_nang_kg"], 1))
        p["chieu_cao_cm"] = clean_num(round(p["chieu_cao_cm"]))
        profile = p
    meals = [{
        "bua": m.bua,
        "mon": m.mon,
        "kcal": round(m.kcal),
        "protein_g": clean_num(round(m.protein_g, 1)),
        "chat_beo_g": clean_num(round(m.chat_beo_g, 1)),
        "carb_g": clean_num(round(m.carb_g, 1)),
        "chat_xo_g": clean_num(round(m.chat_xo_g, 1))
    } for m in b.meals]
    last = None
    if b.last_meal:
        last = [{
            "ten": i.ten,
            "gram": clean_num(round(i.gram)),
            "kcal": round(i.kcal),
            "protein_g": clean_num(round(i.protein_g, 1))
        } for i in b.last_meal]
    rng = random.Random(datetime.date.today().toordinal())
    return build_ctx(profile, meals, FOODS, last, rng)

app = FastAPI(title="SmartFood AI Chatbot Server")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

def check_key(x_api_key: Optional[str]):
    expected = os.environ.get("CHATBOT_API_KEY", "")
    if not expected:
        return
    if not x_api_key or not secrets.compare_digest(x_api_key, expected):
        raise HTTPException(status_code=401, detail="Sai hoặc thiếu API Key (X-API-Key)")

@app.get("/")
@app.get("/health")
def health():
    return {
        "ok": True,
        "service": "SmartFood Chatbot API",
        "knowledge_count": col.count(),
        "device": "cuda" if torch.cuda.is_available() else "cpu"
    }

@app.post("/chat_user")
def chat_user(body: ChatIn, x_api_key: Optional[str] = Header(default=None)):
    check_key(x_api_key)
    try:
        ctx = norm_ctx(body)
        chunks = retrieve(body.question, k=2)
        # Giữ tối đa 4 lượt tin nhắn gần nhất trong history để prompt luôn gọn gàng
        safe_history = body.history[-4:] if body.history else None
        msgs = build_messages(body.question, ctx, chunks, history=safe_history)
        with GPU_LOCK:
            answer = generate(msgs)
        return {
            "answer": answer,
            "canh_bao": ctx.get("canh_bao", []),
            "sources": chunks[:2]
        }
    except Exception as e:
        import traceback
        traceback.print_exc()
        raise HTTPException(status_code=500, detail=str(e))


## Bước 6: Khởi động Server & Mở kết nối ngrok ra ngoài Internet
*(Ô này in ra `CHATBOT_URL = https://xxxx.ngrok-free.app`. Hãy copy URL đó vào file `backend/.env`)*

In [ ]:
import os, time, threading
import uvicorn
from pyngrok import ngrok

def get_secret_or_input(key_name, prompt_msg):
    try:
        from google.colab import userdata
        val = userdata.get(key_name)
        if val: return val.strip()
    except Exception:
        pass
    val = os.environ.get(key_name)
    if val: return val.strip()
    return input(prompt_msg).strip()

CHATBOT_API_KEY = get_secret_or_input("CHATBOT_API_KEY", "Nhập CHATBOT_API_KEY (ví dụ smartfood_secret_key_2026): ")
NGROK_TOKEN = get_secret_or_input("NGROK_TOKEN", "Nhập NGROK_TOKEN (lấy tại https://dashboard.ngrok.com): ")
NGROK_DOMAIN = None
try:
    from google.colab import userdata
    NGROK_DOMAIN = userdata.get("NGROK_DOMAIN")
except Exception:
    pass

os.environ["CHATBOT_API_KEY"] = CHATBOT_API_KEY
ngrok.set_auth_token(NGROK_TOKEN)

# Dọn dẹp các tunnel cũ tránh lỗi trùng lặp (ERR_NGROK_3200)
try:
    for t in ngrok.get_tunnels():
        ngrok.disconnect(t.public_url)
    os.system("pkill ngrok")
    time.sleep(1)
except Exception:
    pass

PORT = 8001
server = uvicorn.Server(uvicorn.Config(app, host="0.0.0.0", port=PORT, log_level="warning"))
thread = threading.Thread(target=server.run, daemon=True)
thread.start()
time.sleep(3)

tunnel = ngrok.connect(PORT, "http", domain=NGROK_DOMAIN) if NGROK_DOMAIN else ngrok.connect(PORT, "http")
PUBLIC_URL = tunnel.public_url

print("=" * 65)
print("🚀 SERVER CHATBOT ĐÃ HOẠT ĐỘNG THÀNH CÔNG!")
print(f"👉 CHATBOT_URL     = {PUBLIC_URL}")
print(f"👉 CHATBOT_API_KEY = {CHATBOT_API_KEY}")
print("=" * 65)
print("Hãy mở file backend/.env trên máy tính và cập nhật:")
print(f"CHATBOT_URL={PUBLIC_URL}")
print(f"CHATBOT_API_KEY={CHATBOT_API_KEY}")
print("=" * 65)


## Bước 7: Kiểm tra nhanh từ chính Colab

In [ ]:
import requests, json

headers = {
    "X-API-Key": os.environ["CHATBOT_API_KEY"],
    "ngrok-skip-browser-warning": "true",
    "Content-Type": "application/json"
}

# 1. Kiểm tra Health
health_res = requests.get(PUBLIC_URL + "/health", headers=headers, timeout=20)
print("1. Kiểm tra /health:")
print(health_res.json())

# 2. Thử hỏi 1 câu
test_body = {
    "question": "Hôm nay mình ăn 1 tô phở và 1 bánh mì thì còn ăn được bao nhiêu kcal?",
    "profile": {
        "tuoi": 22,
        "gioi_tinh": "nu",
        "can_nang_kg": 52,
        "chieu_cao_cm": 160,
        "muc_van_dong": "nhe",
        "muc_tieu": "giam_can"
    },
    "meals": [
        {"bua": "sang", "mon": ["Phở"], "kcal": 450, "protein_g": 25, "chat_beo_g": 12, "carb_g": 60, "chat_xo_g": 3}
    ]
}

print("\n2. Kiểm tra /chat_user (Đang suy luận...):")
chat_res = requests.post(PUBLIC_URL + "/chat_user", json=test_body, headers=headers, timeout=60)
print("Status code:", chat_res.status_code)
print("Kết quả trả về:")
print(json.dumps(chat_res.json(), ensure_ascii=False, indent=2))


## Bước 8: Giữ phiên chạy & Lưu ý khi sử dụng
- Hãy để tab Colab này luôn mở khi đang dùng app hoặc khi demo đồ án.
- Nếu phiên Colab bị ngắt: Bạn chỉ cần vào **Runtime -> Run all** (Chạy lại tất cả) và sao chép `CHATBOT_URL` mới dán vào `backend/.env` là xong!